# 03 — Pré-treino YOLOv8 no dataset público (RWF-2000)

> **Antes do notebook 04** — modelo com classes `person` / `violence` / `pre_violence`  
> treinado só em `data/datasets/public_only/`, sem dados do cliente.

Gera `models/public_only_best.pt`, usado pelo **04** (pré-rotulagem) e como **peso inicial** opcional nos notebooks **06** (cliente) e **07** (combinado).

**Pré-requisito:** `02_dataset_public.ipynb` · **Depois:** `04_client_classification.ipynb`

In [ ]:
!pip install -q ultralytics matplotlib seaborn pandas

In [ ]:
import yaml, json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

DATASET_YAML = Path("../data/datasets/public_only/dataset.yaml")
MODELS_DIR   = Path("../models")
RUNS_DIR     = MODELS_DIR / "runs" / "public_only"   # onde o ultralytics salva os runs
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_YAML.exists(), "Execute 02_dataset_public.ipynb primeiro."

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available(): print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Dataset: {DATASET_YAML}")

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CFG = dict(
    model        = "yolov8m.pt",   # yolov8n para experimento rápido
    epochs       = 100,
    batch        = 16,
    img_size     = 640,
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    patience     = 20,
    optimizer    = "AdamW",
    cls          = 0.7,   # peso maior em classificação
    box          = 7.5,
    dfl          = 1.5,
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.15,
    fliplr       = 0.5,
    degrees      = 10.0,
    save_period  = 10,
)
RUN = f"public_only_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Run: {RUN}")

In [ ]:
model   = YOLO(CFG["model"])
results = model.train(
    data         = str(DATASET_YAML),
    epochs       = CFG["epochs"],
    imgsz        = CFG["img_size"],
    batch        = CFG["batch"],
    device       = DEVICE,
    lr0          = CFG["lr0"],
    lrf          = CFG["lrf"],
    momentum     = CFG["momentum"],
    weight_decay = CFG["weight_decay"],
    warmup_epochs= CFG["warmup_epochs"],
    patience     = CFG["patience"],
    optimizer    = CFG["optimizer"],
    cls          = CFG["cls"],
    box          = CFG["box"],
    dfl          = CFG["dfl"],
    augment      = CFG["augment"],
    mosaic       = CFG["mosaic"],
    mixup        = CFG["mixup"],
    fliplr       = CFG["fliplr"],
    degrees      = CFG["degrees"],
    project      = str(RUNS_DIR),
    name         = RUN,
    save_period  = CFG["save_period"],
    verbose      = True,
)
print("✓ Treinamento concluído")

In [ ]:
run_dir = RUNS_DIR / RUN
csv_p   = run_dir / "results.csv"

if csv_p.exists():
    df = pd.read_csv(csv_p); df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2,3,figsize=(16,8))
    pairs = [("train/box_loss","Train Box Loss","tomato"),
             ("train/cls_loss","Train Cls Loss","darkorange"),
             ("val/box_loss",  "Val Box Loss",  "steelblue"),
             ("metrics/mAP50(B)","mAP@50","seagreen"),
             ("metrics/precision(B)","Precision","slateblue"),
             ("metrics/recall(B)",  "Recall",   "mediumvioletred")]
    for ax,(col,title,color) in zip(axes.flat, pairs):
        if col in df.columns:
            ax.plot(df["epoch"],df[col],color=color,lw=2)
        ax.set_title(title); ax.grid(alpha=0.3)
    plt.suptitle("Treinamento — Public only",fontsize=13)
    plt.tight_layout(); plt.show()

    if "metrics/mAP50(B)" in df.columns:
        best = df["metrics/mAP50(B)"].max()
        print(f"Melhor mAP@50: {best:.4f}")

# Avaliação no teste
best_w = run_dir / "weights" / "best.pt"
if best_w.exists():
    best_m = YOLO(str(best_w))
    metrics = best_m.val(data=str(DATASET_YAML), split="test", device=DEVICE)

    # Salvar métricas para comparação no notebook 08
    result_summary = dict(
        experiment   = "public_only",
        weights      = str(best_w),
        map50        = metrics.box.map50,
        map50_95     = metrics.box.map,
        precision    = metrics.box.mp,
        recall       = metrics.box.mr,
        ap50_per_class = {"person": float(metrics.box.ap50[0]) if len(metrics.box.ap50)>0 else None,
                          "violence": float(metrics.box.ap50[1]) if len(metrics.box.ap50)>1 else None,
                          "pre_violence": float(metrics.box.ap50[2]) if len(metrics.box.ap50)>2 else None},
    )
    with open(MODELS_DIR/"results_public_only.json", "w") as f:
        json.dump(result_summary, f, indent=2)

    print(f"\nmAP@50 (teste): {metrics.box.map50:.4f}")
    print(f"Resultados salvos para comparação no notebook 08.")

    # Copiar melhor peso para models/ (local esperado pelos notebooks 08 e 10)
    import shutil
    shutil.copy(str(best_w), str(MODELS_DIR/"public_only_best.pt"))
    print(f"✓ Pesos: models/public_only_best.pt")

    # Exportar ONNX ao lado dos pesos
    best_m.export(format="onnx", imgsz=CFG["img_size"], simplify=True, opset=17)